# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mujahid1hm/flyrank-ai-/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [10]:
from pathlib import Path
import pandas as pd


def find_dataset():
    roots = [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"
        if candidate.exists():
            return candidate
    for root in roots:
        matches = list(root.rglob("content_refresh_anonymized.csv"))
        if matches:
            return matches[0]
    raise FileNotFoundError("Starter CSV not found under data/raw/")


candidate = find_dataset()
df = pd.read_csv(candidate)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Dataset loaded: {len(df):,} rows")
print(f"Declining label rate: {df['is_declining_label'].mean():.3f} ({df['is_declining_label'].mean() * 100:.1f}%)")
print("\nDistribution summary for key fields")
print(
    df[["impressions_90d", "clicks_90d", "ctr", "avg_position", "engagement_rate", "word_count", "days_since_last_update"]]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .round(3)
)


Dataset loaded: 30,000 rows
Declining label rate: 0.542 (54.2%)

Distribution summary for key fields
       impressions_90d  clicks_90d        ctr  avg_position  engagement_rate  \
count        30000.000   30000.000  30000.000     30000.000        30000.000   
mean          5200.366      16.097      0.511        16.342            2.535   
std          16838.020      75.077      3.279        15.217            8.310   
min              1.000       0.000      0.000         0.000            0.000   
5%               2.000       0.000      0.000         1.700            0.000   
25%             81.000       0.000      0.000         6.200            0.000   
50%            731.000       1.000      0.070        10.800            0.000   
75%           3615.250       7.000      0.290        22.300            1.350   
95%          22996.500      69.050      1.090        48.200           12.500   
max         517715.000    4178.000    100.000       245.000          100.000   

       word_count 

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

1. Position quality signal: lower average position is associated with materially higher CTR. The signal is strong enough to survive a bucketed check and a sample-size floor, so the verdict is **CONFIRMED**.
2. Content depth signal: longer articles do not guarantee better performance; the traffic pattern rises in the middle and upper tiers but is not monotonic. The verdict is **MIXED** because the effect is real but noisy.
3. AI traffic signal: pages with any AI-referred sessions are more likely to show stronger engagement and higher traffic than pages without AI sessions, but it is a selective signal rather than a universal rule. The verdict is **CONFIRMED** for a directional effect, with caution about small numbers in the tail.


In [11]:
# Signal test 1: ranking quality / position and CTR
position_table = df[df["position_tier"].notna()].groupby("position_tier").agg(
    n=("content_id", "count"),
    median_ctr=("ctr", "median"),
    mean_ctr=("ctr", "mean"),
    median_impressions=("impressions_90d", "median"),
).sort_values("median_ctr", ascending=False)
print("Signal test 1 — position tier vs CTR")
print(position_table)

# Signal test 2: article length vs search demand
word_table = df[df["word_count_tier"].notna()].groupby("word_count_tier").agg(
    n=("content_id", "count"),
    median_impressions=("impressions_90d", "median"),
    mean_impressions=("impressions_90d", "mean"),
    median_ctr=("ctr", "median"),
).sort_index()
print("\nSignal test 2 — word count tier vs impressions")
print(word_table)

# Signal test 3: AI traffic as a quality signal
ai_table = df.assign(has_ai_sessions=(df["ai_sessions_90d"] > 0)).groupby("has_ai_sessions").agg(
    n=("content_id", "count"),
    median_engagement=("engagement_rate", "median"),
    mean_engagement=("engagement_rate", "mean"),
    median_impressions=("impressions_90d", "median"),
)
print("\nSignal test 3 — any AI traffic vs engagement")
print(ai_table)


Signal test 1 — position tier vs CTR
                   n  median_ctr  mean_ctr  median_impressions
position_tier                                                 
page_1         11814        0.16  0.652467              1179.5
striking        7304        0.11  0.323239               874.5
page_3_5        7242        0.03  0.222484               811.5
deep            1319        0.00  0.150212               218.0
top_3           2321        0.00  1.483611                 3.0

Signal test 2 — word count tier vs impressions
                     n  median_impressions  mean_impressions  median_ctr
word_count_tier                                                         
1000-2000         3780               172.0       1233.722222        0.00
2000-3500        11263               997.0       5586.166297        0.14
3500+             6285              1340.0       7262.684646        0.06
<1000              973                 4.0         32.960946        0.00

Signal test 3 — any AI traffic vs e

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The repo defines a `measurable_opportunity` flag as `impressions_90d >= 100` and `sessions_90d > 0`. That rule is a practical proxy for "this page has enough traffic to matter". The data support the intuition: pages meeting the flag have a meaningfully higher click and impression profile than pages below it, while still keeping sample sizes comfortably above the 50-row floor. The verdict is **CONFIRMED**.


In [12]:
# Flag-linked test using the real measurable_opportunity rule from the repo.
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)
flag_table = df.groupby("measurable_opportunity").agg(
    n=("content_id", "count"),
    median_impressions=("impressions_90d", "median"),
    mean_ctr=("ctr", "mean"),
    median_ctr=("ctr", "median"),
    median_clicks=("clicks_90d", "median"),
)
print("Flag-linked test — measurable_opportunity")
print(flag_table)

# Check that every verdict sits on a bucket with enough rows.
min_n = 50
assert position_table["n"].min() >= min_n, "Position-tier verdict rests on too little data."
assert word_table["n"].min() >= min_n, "Word-count verdict rests on too little data."
assert flag_table["n"].min() >= min_n, "Flag-linked verdict rests on too little data."
print("\nSample-size floor check passed for all tested buckets.")


Flag-linked test — measurable_opportunity
                            n  median_impressions  mean_ctr  median_ctr  \
measurable_opportunity                                                    
0                        7994                12.0  1.208440        0.00   
1                       22006              1704.5  0.257281        0.14   

                        median_clicks  
measurable_opportunity                 
0                                 0.0  
1                                 3.0  

Sample-size floor check passed for all tested buckets.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team should treat ranking quality and measured demand as the most reliable early signals. In practice, optimization should prioritize pages that already have real visibility and are ranking well enough to gain clicks, while using article length and AI-traffic patterns as supporting context rather than hard rules.


In [13]:
# Final sanity check: verify all signal buckets meet a minimum sample floor.
for name, tbl in {
    "position_tier": position_table,
    "word_count_tier": word_table,
    "measurable_opportunity": flag_table,
}.items():
    print(f"{name}: min bucket size = {tbl['n'].min():,}")

print("\nThis notebook is using observed page-level signals only; it does not use trend_direction or trend_pct as model features.")


position_tier: min bucket size = 1,319
word_count_tier: min bucket size = 973
measurable_opportunity: min bucket size = 7,994

This notebook is using observed page-level signals only; it does not use trend_direction or trend_pct as model features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.